# 02 - Price spreads and net-price opportunities (Q1, Q2)

Q1: how big is the same-day price gap between nearby mandis (<= 100 km road estimate)?  
Q2: after transport costs, how often is selling at a farther mandi profitable, and by how much?

In [1]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
from mandipulse import viz
from mandipulse.viz import COMMODITIES, COMMODITY_COLORS, INK, INK_2, MUTED, load
pd.set_option("display.width", 160)
viz.style()

## Q1 - same-day gap between nearby markets

In [2]:
q1 = load("phase3", "q1_spread_by_commodity")
q1

,commodity,n_pair_days,n_pairs,median_abs_gap_rs_qtl,p90_abs_gap_rs_qtl,median_pct_gap,p90_pct_gap,pct_pair_days_gap_ge_20pct
0,Onion,2331040,3220,150.0,675.0,10.5,55.0,33.1
1,Potato,2419690,2288,100.0,450.0,11.1,56.3,35.5
2,Tomato,2026802,2262,200.0,830.0,15.3,77.4,43.2


In [3]:
dist = load("phase3", "q1_gap_by_distance")
dist["band_mid"] = dist["road_km_band_start"] + 10
fig, ax = viz.figure("Q1. Same-day price gap between market pairs, by distance",
                     "Median gap as % of the lower price, pairs <= 100 km road estimate, days both reported")
for c in COMMODITIES:
    g = dist[dist["commodity"] == c]
    ax.plot(g["band_mid"], g["median_pct_gap"], color=COMMODITY_COLORS[c], marker="o", markersize=5, label=c)
viz.label_line_ends(ax, dist, "band_mid", "median_pct_gap", "commodity", fmt="{:.1f}%")
ax.set_xlabel("road distance between the two markets, km (20 km bands)")
ax.set_ylabel("median gap, % of lower price")
ax.set_ylim(0)
viz.legend(ax)
viz.save(fig, "q1_gap_by_distance", note="Road distance = straight-line distance x 1.3 (assumption).")
dist.pivot(index="road_km_band_start", columns="commodity", values="median_pct_gap")

commodity,Onion,Potato,Tomato
road_km_band_start,,,
0,9.9,10.1,14.3
20,10.5,11.5,15.3
40,9.5,10.0,14.3
60,10.7,11.4,15.3
80,11.1,11.7,16.0


In [4]:
mon = load("phase3", "q1_spread_monthly")
mon["month"] = pd.to_datetime(mon["month"])
fig, ax = viz.figure("Q1. Same-day gap between nearby markets over time",
                     "Monthly median gap as % of the lower price, pairs <= 100 km")
for c in COMMODITIES:
    g = mon[mon["commodity"] == c]
    ax.plot(g["month"], g["median_pct_gap"], color=COMMODITY_COLORS[c], label=c)
viz.label_line_ends(ax, mon, "month", "median_pct_gap", "commodity", fmt="{:.1f}%")
ax.set_ylabel("median gap, %")
ax.set_ylim(0)
viz.legend(ax)
viz.save(fig, "q1_gap_monthly")
mon.groupby("commodity")["median_pct_gap"].describe()

,count,mean,std,min,25%,50%,75%,max
commodity,,,,,,,,
Onion,94.0,11.035106,3.835480,5.4,8.300,10.0,13.275,20.6
Potato,94.0,11.634043,4.593229,4.7,8.025,10.4,14.900,21.3
Tomato,94.0,15.855319,5.934637,5.6,10.875,15.3,18.875,32.0


## Q2 - profitable moves after transport cost

In [5]:
q2 = load("phase3", "q2_opportunity_rate")
scen = ["low", "mid", "high"]
fig, ax = viz.figure("Q2. Market-days with a profitable nearby market, after transport cost",
                     "% of market-days where >= 1 market within 100 km pays >= Rs 100/qtl and >= 5% more, net of transport")
x = np.arange(len(scen))
w = 0.26
for i, c in enumerate(COMMODITIES):
    g = q2[q2["commodity"] == c].set_index("scenario").loc[scen]
    bars = ax.bar(x + (i - 1) * (w + 0.01), g["pct_market_days_with_opportunity"], width=w,
                  color=COMMODITY_COLORS[c], label=c)
    for b, v in zip(bars, g["pct_market_days_with_opportunity"], strict=True):
        ax.annotate(f"{v:.0f}%", (b.get_x() + b.get_width() / 2, v), xytext=(0, 3),
                    textcoords="offset points", ha="center", fontsize=8.5, color=INK_2)
ax.set_xticks(x, ["low cost\n(Rs 1.0/qtl/km + 30)", "mid cost\n(Rs 1.5/qtl/km + 50)", "high cost\n(Rs 2.5/qtl/km + 80)"])
ax.set_ylabel("% of market-days")
ax.set_ylim(0, 70)
viz.legend(ax)
viz.save(fig, "q2_opportunity_rate_by_scenario",
         note="With ~8 neighbours reporting per day, any single neighbour is profitable on only 11-25% of comparisons (see table). "
              "Costs exclude commission and market fees.")
q2

,scenario,commodity,n_market_days,pct_market_days_with_opportunity,median_best_gain_rs_qtl,avg_neighbors_reporting,pct_single_comparisons_opportunity
0,low,Onion,580351,51.0,380.0,8.0,20.5
1,mid,Onion,580351,44.8,381.0,8.0,17.1
2,high,Onion,580351,35.5,392.0,8.0,12.5
3,low,Potato,537622,40.1,291.0,9.0,16.0
4,mid,Potato,537622,33.5,290.0,9.0,12.6
5,high,Potato,537622,24.8,289.0,9.0,8.4
6,low,Tomato,542037,58.5,475.0,7.5,24.0
7,mid,Tomato,542037,53.2,473.0,7.5,20.5
8,high,Tomato,542037,44.6,478.0,7.5,15.7


In [6]:
gd = load("phase3", "q2_gain_distribution")
fig, ax = viz.figure("Q2. Size of the best gain on opportunity days (mid cost)",
                     "Share of opportunity market-days by best net gain, Rs per quintal (bins of 100; last bin = 2000+)")
for c in COMMODITIES:
    g = gd[gd["commodity"] == c].copy()
    g["share"] = 100 * g["n_market_days"] / g["n_market_days"].sum()
    ax.step(g["gain_bin_start_rs_qtl"], g["share"], where="post", color=COMMODITY_COLORS[c], label=c)
ax.set_xlabel("best net gain, Rs / quintal")
ax.set_ylabel("% of opportunity market-days")
ax.set_ylim(0)
viz.legend(ax)
viz.save(fig, "q2_gain_distribution")
gd.pivot(index="gain_bin_start_rs_qtl", columns="commodity", values="n_market_days").head(8)

commodity,Onion,Potato,Tomato
gain_bin_start_rs_qtl,,,
100,54325,53820,45666
200,46027,39197,41111
300,34944,26925,35292
400,25280,18528,28529
500,19211,12040,23580
600,15299,8480,19943
700,11901,5943,15658
800,9687,4552,14194


In [7]:
yr = load("phase3", "q2_opportunity_by_year")
yr.pivot(index="year", columns="commodity", values="pct_market_days_with_opportunity")

commodity,Onion,Potato,Tomato
year,,,
2018,46.1,35.9,51.7
2019,47.5,34.9,63.6
2020,45.4,34.6,52.5
2021,47.1,29.1,51.2
2022,46.0,33.6,57.3
2023,38.6,28.8,48.5
2024,42.9,34.6,50.8
2025,45.7,37.5,48.7


## Sanity checks behind the Q2 number

In [8]:
for q in ["sanity_02_opportunity_concentration", "sanity_04_structural_vs_sporadic",
          "sanity_05_actionable_opportunities", "sanity_06_variety_mix", "sanity_07_opportunity_by_state"]:
    print(q)
    display(load("phase3", q))

sanity_02_opportunity_concentration


,slice,pct_of_opportunities
0,top 10 destination markets,13.5
1,top 5% of destination markets,28.5
2,top 20% of destination markets,65.8


sanity_04_structural_vs_sporadic


,pair_profitable_on,n_directed_pairs,comparison_days,opportunity_days,pct_of_all_opportunity_days
0,0-20% of days,9722,9522859.0,674562.0,30.1
1,20-40% of days,2974,2468379.0,702494.0,31.3
2,40-60% of days,1566,1081446.0,527013.0,23.5
3,60-80% of days,802,423389.0,287045.0,12.8
4,80-100% of days,476,58991.0,50437.0,2.3


sanity_05_actionable_opportunities


,commodity,n_signals,pct_signal_still_opportunity,pct_signal_gain_positive,median_realised_gain_after_signal,median_gain_ex_post
0,Onion,693967,80.3,90.0,286.0,322.0
1,Potato,558411,85.3,94.2,238.0,253.0
2,Tomato,754756,80.5,89.3,321.0,362.0


sanity_06_variety_mix


,pair_type,commodity,n_comparisons,pct_opportunity,median_pct_gap
0,different dominant variety,Onion,903458,24.7,0.194
1,same dominant variety,Onion,3758622,15.3,0.085
2,different dominant variety,Potato,2549820,14.3,0.128
3,same dominant variety,Potato,2289560,10.8,0.086
4,different dominant variety,Tomato,2189384,20.7,0.143
5,same dominant variety,Tomato,1864220,20.3,0.139


sanity_07_opportunity_by_state


,state,commodity,n_market_days,pct_of_commodity_market_days,pct_with_opportunity,avg_neighbors
0,Gujarat,Onion,28945,5.0,35.6,2.4
1,Madhya Pradesh,Onion,63937,11.0,53.7,4.3
2,Maharashtra,Onion,95133,16.4,44.4,7.2
3,Uttar Pradesh,Onion,392336,67.6,44.1,9.3
4,Gujarat,Potato,25827,4.8,43.1,3.2
5,Madhya Pradesh,Potato,45173,8.4,47.5,3.2
6,Maharashtra,Potato,34994,6.5,41.5,2.3
7,Uttar Pradesh,Potato,431628,80.3,30.8,10.5
8,Gujarat,Tomato,40121,7.4,48.3,3.2
9,Madhya Pradesh,Tomato,48111,8.9,54.5,3.1
